# Data Generation

Generates and saves graphs and vectors for CV and test

In [1]:
import numpy as np
import pandas as pd
import json, os, random, pickle
from copy import deepcopy
from types import SimpleNamespace

In [2]:
import torch
import torch.nn.functional as F
from torch.nn import Linear, Dropout, BatchNorm1d, Sequential, ReLU
from torch_geometric.nn import GCNConv, GINEConv, GATv2Conv, global_mean_pool
from torch_geometric.loader import DataLoader

/Users/xgv7306/miniconda/envs/pbsg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import sys
from pathlib import Path
sys.path.append(str(Path('/Users/xgv7306/Library/CloudStorage/OneDrive-NorthwesternUniversity/northwestern/2-PolyID:Dora/PBSG')))


import pbsg
from pbsg.polymer import polymers_from_df
from pbsg.polymer.generators import smiles_from_graphs
from pbsg.ml import to_pyg
from pbsg.ml.pyg import architecture_onehot, classify_stereo
from pbsg.polymer.parser import parse_smiles_list
import ast

def safe_parse(val):
    try:
        if pd.isna(val): return None
    except (TypeError, ValueError): pass
    while isinstance(val, str):
        try: val = ast.literal_eval(val)
        except (ValueError, SyntaxError): break
    return val

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [10]:
df = pd.read_csv('data_classifier.csv')
for col in ['beads_list', 'lengths', 'reactivity_ratios', 'spatiality_rules']:
    if col in df.columns:
        df[col] = df[col].apply(safe_parse)
df['lengths']             = df['lengths'].apply(lambda x: None if isinstance(x, str) else x)
df['architecture_onehot'] = df.apply(architecture_onehot, axis=1)
df['beads_list']          = df['repeat_units'].apply(parse_smiles_list)
df['stereo_class']= df.apply(classify_stereo, axis=1)

df.to_csv('data_classifier.csv', index=False)

In [5]:
set_seed(42)
bead_polymers = polymers_from_df(df, total_chain_length=40)

graphs = []
for i, polymer in enumerate(bead_polymers):
    try:
        if type(polymer.has_Tm) == float:
            continue
        data = to_pyg(polymer, stereo=True)
        if data is None:
            continue
        data.y         = torch.tensor([1.0 if polymer.has_Tm is True else 0.0], dtype=torch.float32)
        data.x         = data.x.float()
        data.edge_attr = data.edge_attr.float()
        graphs.append(data)
    except Exception as e:
        print(f"Error at index {i}: {e}")

print(f"Graphs built: {len(graphs)}")
print(f"  x shape:         {graphs[0].x.shape}")
print(f"  edge_attr shape: {graphs[0].edge_attr.shape}")
print(f"  architecture:    {graphs[0].architecture.shape}")

IN_CHANNELS = graphs[0].x.shape[1]
EDGE_DIM    = graphs[0].edge_attr.shape[1]
META_DIM    = graphs[0].architecture.shape[1]
print(f"\nin_channels={IN_CHANNELS}  edge_dim={EDGE_DIM}  meta_dim={META_DIM}")


Graphs built: 643
  x shape:         torch.Size([40, 85])
  edge_attr shape: torch.Size([78, 1])
  architecture:    torch.Size([1, 13])

in_channels=85  edge_dim=1  meta_dim=13


In [6]:
set_seed(42)
polymers = smiles_from_graphs(df, total_chain_length=40)
polymers['architecture_onehot'] = df['architecture_onehot']

def build_flat_features_poly(graphs, df, fp_bits=1024):
    """
    FP + pooled physical descriptors + metadata vector + pooled edge features.

    Node layout:
        [0:32]   morgan PCA    [32:64] rdkit PCA
        [64:82]  physical desc (18)    [82:85] side one-hot (3, excluded)
    data.architecture : [1, META_DIM]  arch(4)+stereo(6)+Pm(1)+x1(1) = 12
    data.edge_attr    : [n_edges, edge_dim]
    """
    from rdkit import Chem
    from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator, FingerprintGenerator32
    _gen = GetMorganGenerator(radius=2, fpSize=fp_bits, includeChirality=True)
    DESC_START, DESC_END = 64, 82

    X_fp_list, X_pooled_list = [], []
    for g, (_, row) in zip(graphs, df.iterrows()):
        pooled_desc = g.x.float().numpy()[:, DESC_START:DESC_END].mean(axis=0)
        meta        = g.architecture.float().numpy().flatten()
        ea = g.edge_attr
        pooled_edge = (ea.float().numpy().mean(axis=0)
                       if ea is not None and ea.numel() > 0
                       else np.zeros(ea.shape[1] if ea is not None else 1, dtype=np.float32))
        mol = Chem.MolFromSmiles(str(row["p_smiles"]))
        fp  = (np.array(_gen.GetFingerprintAsNumPy(mol), dtype=np.float32)
               if mol is not None else np.zeros(fp_bits, dtype=np.float32))
        X_fp_list.append(fp)
        X_pooled_list.append(np.concatenate([fp, pooled_desc, meta, pooled_edge]))

    X_fp        = np.array(X_fp_list)
    X_fp_pooled = np.array(X_pooled_list)
    y           = df["has_Tm"].astype(int).values
    print(f"X_fp={X_fp.shape}  X_fp_pooled={X_fp_pooled.shape}")
    print(f"  breakdown: {fp_bits} FP + {DESC_END-DESC_START} desc "
          f"+ {meta.shape[0]} meta + {pooled_edge.shape[0]} edge")
    return X_fp, X_fp_pooled, y

X_fp_poly, X_fp_pooled_poly, y_poly = build_flat_features_poly(graphs, polymers)

X_fp=(643, 1024)  X_fp_pooled=(643, 1056)
  breakdown: 1024 FP + 18 desc + 13 meta + 1 edge


In [7]:
np.save("Vectors/X_fp_poly.npy", X_fp_poly)
np.save("Vectors/X_fp_pooled_poly.npy", X_fp_pooled_poly)

In [8]:
np.save("Vectors/y.npy", y_poly)

In [9]:
torch.save(graphs, "Graphs/graphs_PBSG.pt")

In [10]:
def build_flat_features_RU(graphs, bead_polymers, fp_bits=1024):
    """
    FP + pooled physical descriptors + metadata + pooled edge features.
    Node layout: [0:32] morgan PCA, [32:64] rdkit PCA, [64:82] descriptors, [82:85] side
    data.architecture: [1, META_DIM] = arch(4)+stereo(6)+Pm(1)+Px(1)+asym(1)
    """
    from rdkit import Chem
    from functools import reduce
    from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
    _gen = GetMorganGenerator(radius=2, fpSize=fp_bits, includeChirality=True)
    DESC_START, DESC_END = 64, 82

    X_fp_list, X_pooled_list = [], []
    for g, beads in zip(graphs, bead_polymers):
        pooled_desc = g.x.float().numpy()[:, DESC_START:DESC_END].mean(axis=0)
        meta        = g.architecture.float().numpy().flatten()
        ea          = g.edge_attr
        pooled_edge = (ea.float().numpy().mean(axis=0)
                       if ea is not None and ea.numel() > 0
                       else np.zeros(ea.shape[1] if ea is not None else 1, dtype=np.float32))
        smis=[]
        for b in beads.nodes:
            smi=b.bead_id
            smis.append(smi)
        u_smis=list(set(smis))
        mols = tuple([Chem.MolFromSmiles(s) for s in u_smis])
        fps=[]
        for m in mols:
            fp  = (np.array(_gen.GetFingerprintAsNumPy(m), dtype=np.float32)
                    if m is not None else np.zeros(fp_bits, dtype=np.float32))
            fps.append(fp)
        #print(type(fps[0]))
        fp=np.mean(np.array(fps), axis=0)
        #print(fp)

        X_fp_list.append(fp)
        X_pooled_list.append(np.concatenate([fp, pooled_desc, meta, pooled_edge]))

    X_fp        = np.array(X_fp_list)
    X_fp_pooled = np.array(X_pooled_list)
    y           = df["has_Tm"].astype(int).values
    print(f"X_fp_pooled={X_fp_pooled.shape}  y={y.shape}")
    return X_fp, X_fp_pooled, y

X_fp_RU, X_fp_pooled_RU, y_RU = build_flat_features_RU(graphs, bead_polymers)

X_fp_pooled=(643, 1056)  y=(643,)


In [11]:
np.save("Vectors/X_fp_RU.npy", X_fp_RU)
np.save("Vectors/X_fp_pooled_RU.npy", X_fp_pooled_RU)

In [12]:
from pbsg.ml.m2g import mol_to_graph
from rdkit import Chem

smi_graphs=[]
for p in polymers['p_smiles']:
    mol=Chem.MolFromSmiles(p)
    g=mol_to_graph(mol)
    smi_graphs.append(g)

for g, (_, row) in zip(smi_graphs, df.iterrows()):
    g.y=torch.tensor([float(row['has_Tm'])], dtype=torch.float32)

In [13]:
torch.save(smi_graphs, "Graphs/graphs_SMILES.pt")

In [14]:
for g_sm, g in zip(smi_graphs, graphs):
    g_sm.architecture=g.architecture

In [15]:
torch.save(smi_graphs, "Graphs/graphs_SMILES+global.pt")